###1. Installing packages for Elastic Search and ColBERT

In [ ]:
!pip install pytrec-eval-terrier -q
!pip install elasticsearch==7.9.1 -q
!pip uninstall -y numpy transformers ragatouille faiss-cpu faiss-gpu-cu12 faiss-gpu -q
!pip install -q transformers==4.44.0
!pip install -q ragatouille==0.0.9
!pip install faiss-gpu-cu12 -q
!pip install "numpy<2.0"
print("Done — RESTART SESSION NOW, then continue from next cell")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.2/219.2 kB 17.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 139.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 132.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 4.3 MB/s eta 0:00:00
   

Done — RESTART SESSION NOW, then continue from next cell


In [ ]:
files = [
    '/usr/local/lib/python3.12/dist-packages/ragatouille/models/index.py',
    '/usr/local/lib/python3.12/dist-packages/colbert/indexer.py',
    '/usr/local/lib/python3.12/dist-packages/colbert/infra/launcher.py',
    '/usr/local/lib/python3.12/dist-packages/colbert/indexing/collection_indexer.py',
    '/usr/local/lib/python3.12/dist-packages/colbert/utils/runs.py',
]
for filepath in files:
    try:
        with open(filepath, 'r') as f:
            content = f.read()
        content = content.replace('time.sleep(5)', 'pass')
        content = content.replace('time.sleep(20)', 'pass')
        content = content.replace(
            'print("Will continue with CPU indexing in 5 seconds...")\n                    pass',
            'pass'
        )
        with open(filepath, 'w') as f:
            f.write(content)
        print(f'Patched: {filepath}')
    except Exception as e:
        print(f'Skip: {filepath} — {e}')
print('All done')

Patched: /usr/local/lib/python3.12/dist-packages/ragatouille/models/index.py
Patched: /usr/local/lib/python3.12/dist-packages/colbert/indexer.py
Patched: /usr/local/lib/python3.12/dist-packages/colbert/infra/launcher.py
Patched: /usr/local/lib/python3.12/dist-packages/colbert/indexing/collection_indexer.py
Patched: /usr/local/lib/python3.12/dist-packages/colbert/utils/runs.py
All done


In [ ]:
!wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-oss-7.9.2-linux-x86_64.tar.gz
!tar -xzf elasticsearch-oss-7.9.2-linux-x86_64.tar.gz
!chown -R daemon:daemon elasticsearch-7.9.2

In [ ]:
import subprocess
subprocess.Popen(['sudo', '-u', 'daemon', './elasticsearch-7.9.2/bin/elasticsearch'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('Elasticsearch starting...')

Elasticsearch starting...


In [ ]:
import time, requests
time.sleep(30)
print(requests.get('http://localhost:9200').json())

{'name': 'a141216dd671', 'cluster_name': 'elasticsearch', 'cluster_uuid': 'HBvYnZ7JReaPouzBHCzAHA', 'version': {'number': '7.9.2', 'build_flavor': 'oss', 'build_type': 'tar', 'build_hash': 'd34da0ea4a966c4e49417f2da2f244e3e97b4e6e', 'build_date': '2020-09-23T00:45:33.626720Z', 'build_snapshot': False, 'lucene_version': '8.6.2', 'minimum_wire_compatibility_version': '6.8.0', 'minimum_index_compatibility_version': '6.0.0-beta1'}, 'tagline': 'You Know, for Search'}


In [ ]:
import json, csv, re
from elasticsearch import Elasticsearch, NotFoundError
from elasticsearch.helpers import bulk

es = Elasticsearch("http://localhost:9200")
print('ES working' if es.ping() else 'ES not working')

ES working


###2. Uploading FiQA files

In [ ]:
import shutil
shutil.rmtree('/content/drive')

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
CORPUS_PATH  = "corpus.jsonl"
QUERIES_PATH = "queries.jsonl"

TRAIN_QRELS = "train.tsv"
DEV_QRELS   = "dev.tsv"
TEST_QRELS  = "test.tsv"

In [ ]:
import re
from html.parser import HTMLParser

# HTML stripper
class HTMLStripper(HTMLParser):
    def __init__(self):
        super().__init__()
        self.reset()
        self.fed = []
    def handle_data(self, d):
        self.fed.append(d)
    def get_data(self):
        return ' '.join(self.fed)

def strip_html(text):
    s = HTMLStripper()
    s.feed(text)
    return s.get_data()

# indices / special cases that contain punctuationI
INDEX_MAP = [
    (r"\bS&P\s*500\b",   "sp500"),
    (r"\bS&P\b",         "sandp"),
    (r"\bDow\s*Jones\b", "dowjones"),
    (r"\bNASDAQ\b",      "nasdaq"),
    (r"\bNYSE\b",        "nyse"),
    (r"\bFTSE\b",        "ftse"),
    (r"\bDAX\b",         "dax"),
    (r"\bNikkei\b",      "nikkei"),
]

# Financial ratios — detected before lowercasing
RATIO_MAP = [
    (r"\bP/FCF\b",       "pfcf ratio"),
    (r"\bEV/EBITDA\b",   "ev ebitda ratio"),
    (r"\bP/E\b",         "pe ratio"),
    (r"\bP/B\b",         "pb ratio"),
    (r"\bP/S\b",         "ps ratio"),
    (r"\bROIC\b",        "roic"),
    (r"\bROE\b",         "roe"),
    (r"\bROA\b",         "roa"),
    (r"\bEPS\b",         "eps"),
    (r"\bDPS\b",         "dps"),
    (r"\bNAV\b",         "nav"),
    (r"\bAUM\b",         "aum"),
    (r"\bYTD\b",         "ytd"),
    (r"\bYOY\b",         "yoy"),
    (r"\bQOQ\b",         "qoq"),
    (r"\bMOM\b",         "mom"),
    (r"\bIPO\b",         "ipo"),
    (r"\bETF\b",         "etf"),
    (r"\bREIT\b",        "reit"),
    (r"\bCDO\b",         "cdo"),
    (r"\bCDS\b",         "cds"),
    (r"\bMBS\b",         "mbs"),
]
# $TICKER into bare ticker
TICKER_RE = re.compile(r"\$([A-Z]{1,5})\b")

# Currency and unit symbols into text
# A leading space is inserted so "10%" becomes "10 percent" not "10percent"
CURRENCY_MAP = [
    (r"\$",  "usd"),
    (r"€",   "eur"),
    (r"£",   "gbp"),
    (r"¥",   "jpy"),
    (r"₹",   "inr"),
    (r"₩",   "krw"),
    (r"₿",   "btc"),
    (r"Ξ",   "eth"),
    (r"%",   "percent"),
]



def normalise_financial(text):
    text = TICKER_RE.sub(r'\1', text)                          # $AAPL → AAPL
    for pattern, repl in INDEX_MAP:                            # S&P 500 → sp500
        text = re.sub(pattern, repl, text, flags=re.IGNORECASE)
    for pattern, repl in RATIO_MAP:                            # P/E → pe ratio
        text = re.sub(pattern, repl, text)
    for pattern, repl in CURRENCY_MAP:                         # $ → usd
        text = re.sub(pattern, f' {repl} ', text)
    return text

def clean_text(text):
    if not text:
        return ''
    text = strip_html(text)           # 1. strip HTML
    text = normalise_financial(text)  # 2. financial normalisation
    text = text.lower()               # 3. lowercase
    text = re.sub(r"[^a-z0-9\s']", ' ', text)   # 4. remove special chars
    text = re.sub(r"(?<![a-z])'|'(?![a-z])", ' ', text) # remove stray apostrophes
    text = re.sub(r'\s+', ' ', text).strip()   # 5. collapse whitespace
    return text


###4. Loading and Cleaning the FiQA files

In [ ]:
import json
import csv

# Corpus
with open(CORPUS_PATH, 'r') as f:
    corpus_raw = [json.loads(line) for line in f]

seen, corpus_clean, dupes = set(), [], 0

for doc in corpus_raw:
    doc_id = doc.get('_id') or doc.get('id')
    body = clean_text(doc.get('text', '').strip())

    if body in seen:
        dupes += 1
        continue

    seen.add(body)
    corpus_clean.append({
        'doc_id': doc_id,
        'text': body
    })

print(f'Duplicates removed: {dupes}')
print(f'Clean corpus size:  {len(corpus_clean)}')
print(f'\nExample doc:')
print(json.dumps(corpus_clean[0], indent=2))

Duplicates removed: 66
Clean corpus size:  57572

Example doc:
{
  "doc_id": "3",
  "text": "i'm not saying i don't like the idea of on the job training too but you can't expect the company to do that training workers is not their job they're building software perhaps educational systems in the u s or their students should worry a little about getting marketable skills in exchange for their massive investment in education rather than getting out with thousands in student debt and then complaining that they aren't qualified to do anything"
}


In [ ]:
# Queries
with open(QUERIES_PATH, 'r') as f:
    queries_raw = [json.loads(line) for line in f]

seen, queries_clean, dupes = set(), [], 0

for doc in queries_raw:
    doc_id = doc.get('_id') or doc.get('id')
    body = clean_text(doc.get('text', '').strip())

    if body in seen:
        dupes += 1
        continue

    seen.add(body)
    queries_clean.append({
        'doc_id': doc_id,
        'text': body
    })

print(f'Duplicates removed: {dupes}')
print(f'Clean corpus size:  {len(queries_clean)}')
print(f'\nExample doc:')
print(json.dumps(queries_clean[0], indent=2))

Duplicates removed: 3
Clean corpus size:  6645

Example doc:
{
  "doc_id": "0",
  "text": "what is considered a business expense on a business trip"
}


In [ ]:
# Qrels
QRELS_SPLITS = {
    'train': TRAIN_QRELS,
    'dev':   DEV_QRELS,
    'test':  TEST_QRELS,
}

qrels_all = []

for split, path in QRELS_SPLITS.items():
    with open(path, 'r') as f:
        reader = csv.DictReader(f, delimiter='\t')
        rows = []
        for row in reader:
            rows.append({
                'query_id':  row['query-id'],
                'corpus_id': row['corpus-id'],
                'score':     int(row['score']),
                'split':     split,
            })
        qrels_all.extend(rows)
        print(f'{split}.tsv: {len(rows)} rows')

print(f'\nTotal qrels: {len(qrels_all)}')
print(f'\nExample row:')
print(json.dumps(qrels_all[0], indent=2))

train.tsv: 14166 rows
dev.tsv: 1238 rows
test.tsv: 1706 rows

Total qrels: 17110

Example row:
{
  "query_id": "0",
  "corpus_id": "18850",
  "score": 1,
  "split": "train"
}


###5. Creating BM25 Index

In [ ]:

index_name = 'fiqa'

request_body = {
    'settings': {
        'number_of_shards': 1,
        'number_of_replicas': 0,
        'similarity': {
            'default': {
                'type': 'BM25',
                'b':  0.9,
                'k1': 0.4,
            }
        }
    },
    'mappings': {
        'properties': {
            'text': {'type': 'text'},
        }
    }
}

try:
    es.indices.get(index_name)
    print(f'Index "{index_name}" already exists — deleting it')
    es.indices.delete(index_name)
except NotFoundError:
    pass

print(f'Creating index "{index_name}"')
es.indices.create(index_name, body=request_body)

actions = []
for doc in corpus_clean:
    actions.append({
        '_index': index_name,
        '_id': doc['doc_id'],
        '_source': {'text': doc['text']}
    })
bulk(es, actions)
print(f'Indexed {len(actions)} docs')
print('Done')

###6. BM25 Evaluation

In [ ]:
import pytrec_eval

# Build qrels_dict from test split only
qrels_dict = {}
for row in qrels_all:
    if row['split'] != 'test':
        continue
    qid = row['query_id']
    did = row['corpus_id']
    if qid not in qrels_dict:
        qrels_dict[qid] = {}
    qrels_dict[qid][did] = int(row['score'])

# Run BM25 search for every query and collect results
# bm25_results = {}
# for query in queries_clean:
#     qid = query['doc_id']
#     if qid not in qrels_dict:
#         continue
#     res = es.search(index=index_name, body={
#         'size': 100,
#         'query': {'match': {'text': query['text']}}
#     })
#     bm25_results[qid] = {
#         hit['_id']: hit['_score'] for hit in res['hits']['hits']
#     }


In [ ]:
import pickle

with open('bm25_results.pkl', 'rb') as f:
    bm25_results = pickle.load(f)
print(f'Loaded results for {len(bm25_results)} queries')

Loaded results for 648 queries


In [ ]:
# MRR@10
def mrr_at_10(results, qrels):
    scores = []
    for qid, docs in results.items():
        if qid not in qrels:
            continue
        ranked = sorted(docs.items(), key=lambda x: x[1], reverse=True)[:10]
        rr = 0.0
        for rank, (doc_id, _) in enumerate(ranked, start=1):
            if doc_id in qrels[qid] and qrels[qid][doc_id] > 0:
                rr = 1.0 / rank
                break
        scores.append(rr)
    return sum(scores) / len(scores) if scores else 0.0

# Evaluate
metrics = {"ndcg_cut.10", "recall.100"}
evaluator = pytrec_eval.RelevanceEvaluator(qrels_dict, metrics)
eval_results = evaluator.evaluate(bm25_results)

print(f"nDCG@10:    {sum(r['ndcg_cut_10'] for r in eval_results.values()) / len(eval_results):.4f}")
print(f"MRR@10:     {mrr_at_10(bm25_results, qrels_dict):.4f}")
print(f"Recall@100: {sum(r['recall_100']  for r in eval_results.values()) / len(eval_results):.4f}")

nDCG@10:    0.2288
MRR@10:     0.2820
Recall@100: 0.4997


In [ ]:
import pickle

with open('/content/drive/MyDrive/bm25_results.pkl', 'rb') as f:
    bm25_results = pickle.load(f)
print(f'Loaded results for {len(bm25_results)} queries')

Loaded results for 648 queries


###7. ColBERT

In [ ]:
!pip install ragatouille==0.0.8 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.7/86.7 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.7/491.7 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.1/303.1 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
faiss-gpu-cu12 1.14.1.post1 requires numpy<3,>=2, but you have numpy 

In [ ]:
# Fixes in the environment for COlBERT to run
import sys

# Remove cached modules
for mod in list(sys.modules.keys()):
    if 'ragatouille' in mod or 'colbert' in mod:
        del sys.modules[mod]

# Patch files
files = [
    '/usr/local/lib/python3.12/dist-packages/ragatouille/models/index.py',
    '/usr/local/lib/python3.12/dist-packages/colbert/indexer.py',
    '/usr/local/lib/python3.12/dist-packages/colbert/infra/launcher.py',
    '/usr/local/lib/python3.12/dist-packages/colbert/indexing/collection_indexer.py',
    '/usr/local/lib/python3.12/dist-packages/colbert/utils/runs.py',
]
for filepath in files:
    try:
        with open(filepath, 'r') as f:
            content = f.read()
        content = content.replace('time.sleep(5)', 'pass')
        content = content.replace('time.sleep(20)', 'pass')
        content = content.replace(
            'print("Will continue with CPU indexing in 5 seconds...")\n                    pass',
            'pass'
        )
        with open(filepath, 'w') as f:
            f.write(content)
        print(f'Patched: {filepath}')
    except Exception as e:
        print(f'Skip: {filepath} — {e}')

# Clear pyc cache
import subprocess
subprocess.run(['find', '/usr/local/lib/python3.12/dist-packages/ragatouille', '-name', '*.pyc', '-delete'])
subprocess.run(['find', '/usr/local/lib/python3.12/dist-packages/colbert', '-name', '*.pyc', '-delete'])

print('All done — now run the indexing cell')

In [ ]:
from ragatouille import RAGPretrainedModel

colbert_documents    = [doc['text']   for doc in corpus_clean]
colbert_document_ids = [doc['doc_id'] for doc in corpus_clean]

RAG = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")
RAG.index(
    collection=colbert_documents,
    document_ids=colbert_document_ids,
    index_name="fiqa_colbert",
    max_document_length=180,
    split_documents=False,
    use_faiss=True
)
print("ColBERT indexing complete")

________________________________________________________________________________
WARNING! You have a GPU available, but only `faiss-cpu` is currently installed.
 This means that indexing will be slow. To make use of your GPU.
Please install `faiss-gpu` by running:
pip uninstall --y faiss-cpu & pip install faiss-gpu
 ________________________________________________________________________________


[Apr 06, 17:26:25] #> Note: Output directory .ragatouille/colbert/indexes/fiqa_colbert already exists


[Apr 06, 17:26:25] #> Will delete 1 files already at .ragatouille/colbert/indexes/fiqa_colbert in 20 seconds...
[Apr 06, 17:26:27] [0] 		 #> Encoding 42055 passages..


KeyboardInterrupt: 

###8. ColBERT Evaluation

In [ ]:
colbert_results = {qid: docs for qid, docs in colbert_results.items() if qid in qrels_dict}
print(f'Filtered ColBERT results: {len(colbert_results_test)} queries')

Filtered ColBERT results: 648 queries


In [ ]:
import pytrec_eval

def mrr_at_10(results, qrels):
    scores = []
    for qid, docs in results.items():
        if qid not in qrels:
            continue
        ranked = sorted(docs.items(), key=lambda x: x[1], reverse=True)[:10]
        rr = 0.0
        for rank, (doc_id, _) in enumerate(ranked, start=1):
            if doc_id in qrels[qid] and qrels[qid][doc_id] > 0:
                rr = 1.0 / rank
                break
        scores.append(rr)
    return sum(scores) / len(scores) if scores else 0.0

metrics = {"ndcg_cut.10", "recall.100"}
evaluator = pytrec_eval.RelevanceEvaluator(qrels_dict, metrics)
eval_results = evaluator.evaluate(colbert_results)

print(f"nDCG@10:    {sum(r['ndcg_cut_10'] for r in eval_results.values()) / len(eval_results):.4f}")
print(f"MRR@10:     {mrr_at_10(colbert_results, qrels_dict):.4f}")
print(f"Recall@100: {sum(r['recall_100']   for r in eval_results.values()) / len(eval_results):.4f}")

nDCG@10:    0.3362
MRR@10:     0.4103
Recall@100: 0.6239


In [ ]:
import pickle

with open('/content/drive/MyDrive/colbert_results_fiqa.pkl', 'wb') as f:
    pickle.dump(colbert_results, f)
print('ColBERT results saved to Drive')

ColBERT results saved to Drive


###8. RRF

In [ ]:
def rrf(results_list, k=60):
    fused = {}
    for results in results_list:
        ranked = sorted(results.items(), key=lambda x: x[1], reverse=True)
        for rank, (doc_id, _) in enumerate(ranked, start=1):
            if doc_id not in fused:
                fused[doc_id] = 0
            fused[doc_id] += 1 / (k + rank)
    return fused

# Combine BM25 and ColBERT results
rrf_results = {}
for qid in qrels_dict:
    bm25 = bm25_results.get(qid, {})
    colbert = colbert_results.get(qid, {})
    rrf_results[qid] = rrf([bm25, colbert])

# Evaluate
metrics = {"ndcg_cut.10", "recall.100"}
evaluator = pytrec_eval.RelevanceEvaluator(qrels_dict, metrics)
eval_results = evaluator.evaluate(rrf_results)

print(f"nDCG@10:    {sum(r['ndcg_cut_10'] for r in eval_results.values()) / len(eval_results):.4f}")
print(f"MRR@10:     {mrr_at_10(rrf_results, qrels_dict):.4f}")
print(f"Recall@100: {sum(r['recall_100']   for r in eval_results.values()) / len(eval_results):.4f}")

nDCG@10:    0.2960
MRR@10:     0.3523
Recall@100: 0.6119


In [ ]:
import pickle

with open('/content/drive/MyDrive/rrf_results_fiqa.pkl', 'wb') as f:
    pickle.dump(rrf_results, f)
print('RRF results saved to Drive')

RRF results saved to Drive


###9. Final Evaluation

In [ ]:
def compute_mrr_at_10(results, qrels_dict):
    mrr_scores = []
    for qid in results:
        relevant = set(d for d, s in qrels_dict.get(qid, {}).items() if s > 0)
        ranked_docs = list(results[qid].keys())[:10]
        rr = 0.0
        for rank, doc_id in enumerate(ranked_docs, 1):
            if doc_id in relevant:
                rr = 1.0 / rank
                break
        mrr_scores.append(rr)
    return mrr_scores

metrics = {"ndcg_cut.10", "recall.100"}
evaluator = pytrec_eval.RelevanceEvaluator(qrels_dict, metrics)

bm25_eval    = evaluator.evaluate(bm25_results)
colbert_eval = evaluator.evaluate(colbert_results)
rrf_eval     = evaluator.evaluate(rrf_results)

bm25_ndcg    = [bm25_eval[q]["ndcg_cut_10"]    for q in bm25_eval]
bm25_mrr     = compute_mrr_at_10(bm25_results,    qrels_dict)
bm25_recall  = [bm25_eval[q]["recall_100"]     for q in bm25_eval]

colbert_ndcg   = [colbert_eval[q]["ndcg_cut_10"]  for q in colbert_eval]
colbert_mrr    = compute_mrr_at_10(colbert_results,  qrels_dict)
colbert_recall = [colbert_eval[q]["recall_100"]   for q in colbert_eval]

rrf_ndcg   = [rrf_eval[q]["ndcg_cut_10"]  for q in rrf_eval]
rrf_mrr    = compute_mrr_at_10(rrf_results,  qrels_dict)
rrf_recall = [rrf_eval[q]["recall_100"]   for q in rrf_eval]

print("=" * 62)
print(f"{'Method':<12} {'nDCG@10':>10} {'MRR@10':>10} {'Recall@100':>12}")
print("=" * 62)
print(f"{'BM25':<12} {sum(bm25_ndcg)/len(bm25_ndcg):>10.4f} {sum(bm25_mrr)/len(bm25_mrr):>10.4f} {sum(bm25_recall)/len(bm25_recall):>12.4f}")
print(f"{'ColBERT':<12} {sum(colbert_ndcg)/len(colbert_ndcg):>10.4f} {sum(colbert_mrr)/len(colbert_mrr):>10.4f} {sum(colbert_recall)/len(colbert_recall):>12.4f}")
print(f"{'RRF Hybrid':<12} {sum(rrf_ndcg)/len(rrf_ndcg):>10.4f} {sum(rrf_mrr)/len(rrf_mrr):>10.4f} {sum(rrf_recall)/len(rrf_recall):>12.4f}")
print("=" * 62)

Method          nDCG@10     MRR@10   Recall@100
BM25             0.2288     0.2820       0.4997
ColBERT          0.3362     0.4103       0.6239
RRF Hybrid       0.2960     0.2820       0.6119


In [ ]:
import json
from pathlib import Path

TOP_N_UI = 10
OUT_PATH = "/content/drive/MyDrive/fiqa_demo_results.json"

corpus_lookup = {doc['doc_id']: {'text': doc['text']} for doc in corpus_clean}
queries_lookup = {q['doc_id']: q['text'] for q in queries_clean}

def make_snippet(doc, max_chars=220):
    text = doc.get('text', '')
    return text[:max_chars].rstrip() + ('...' if len(text) > max_chars else '')

def build_method_results(method_results, query_id, corpus_lookup, top_n=10):
    ranked_items = list(method_results.get(query_id, {}).items())[:top_n]
    output = []
    for rank, (doc_id, score) in enumerate(ranked_items, start=1):
        doc = corpus_lookup.get(doc_id) or corpus_lookup.get(str(doc_id)) or {}
        output.append({
            'rank': rank,
            'doc_id': str(doc_id),
            'score': float(score),
            'snippet': make_snippet(doc)
        })
    return output

demo_data = {}
for qid, query_text in queries_lookup.items():
    if qid not in qrels_dict:
        continue
    demo_data[query_text] = {
        'query_id': str(qid),
        'bm25':    build_method_results(bm25_results,    qid, corpus_lookup, top_n=TOP_N_UI),
        'colbert': build_method_results(colbert_results, qid, corpus_lookup, top_n=TOP_N_UI),
        'rrf':     build_method_results(rrf_results,     qid, corpus_lookup, top_n=TOP_N_UI),
    }

Path(OUT_PATH).parent.mkdir(parents=True, exist_ok=True)
with open(OUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(demo_data, f, indent=2, ensure_ascii=False)

print(f'Saved demo JSON to: {OUT_PATH}')
print(f'Total queries saved: {len(demo_data):,}')

Saved demo JSON to: /content/drive/MyDrive/fiqa_demo_results.json
Total queries saved: 648
